# Measles infection in the 1970 British Cohort Study

## Make figures

In [ ]:
# ── Define outcome groups ────
cardiometabolic_binary     <- c("cmvigg")

cardiometabolic_continuous <- c("bmi_z", "bp_dia_z", "bp_sys_z", "chol_std",
                                 "hba1c_std", "hdl_std", "igf1_std", "log_crp_std",
                                 "log_frtin_std", "log_trig_std", "rbc_std", "waisthip_z")

cognitive                  <- c("animal_naming46_z", "letter_correct46_z",
                                 "letter_speed46_z", "vocab42_z", "literacy35_z", "numeracy35_z", "maths_score10_z",
                                 "maths_score16_z", "matrices_score16_z",
                                 "reading_score_raw10_z", "reading_score_total16_z",
                                 "wordlist_delayed46_z", "wordlist_immediate46_z")

cognitive_order <- c(
  # Age 46 (bottom)
  "Age 46 - Processing (speed)",
  "Age 46 - Processing (score)",
  "Age 46 - Verbal Fluency",
  "Age 46 - Verbal Memory (immediate)",
  "Age 46 - Verbal Memory (delayed)",
  # Age 42
  "Age 42 - Vocabulary",
  # Age 35
  "Age 35 - Literacy",
  "Age 35 - Numeracy",
  # Age 16
  "Age 16 - Matrices",
  "Age 16 - Maths",
  "Age 16 - Reading",
  # Age 10 (top)
  "Age 10 - Maths",
  "Age 10 - Reading"
)

In [ ]:
library(ggplot2)
library(dplyr)
library(forcats)
library(patchwork)
library(cowplot)

# ── Custom theme  ─────────────────────────────────
my_custom_theme <- function(base_size = 12, base_family = "Helvetica") {
  theme_minimal(base_size = base_size, base_family = base_family) +
    theme(
      plot.title       = element_text(size = base_size + 4, face = "bold", hjust = 0.5),
      axis.title       = element_text(size = base_size + 2),
      axis.text        = element_text(size = base_size, color = "black"),
      axis.ticks       = element_line(color = "black"),
      panel.grid       = element_blank(),
      panel.background = element_blank(),
      plot.background  = element_blank(),
      panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.8),
      legend.position  = "bottom",
      legend.title     = element_text(face = "bold"),
      legend.text      = element_text(size = base_size),
      strip.text       = element_text(face = "bold", size = base_size)
    )
}

# ── Import saved results ──────────────────────────────────────────────────────
comparison_table <- read.csv("crude_vs_ipw_table_adulteduc.csv", stringsAsFactors = FALSE)

# ── Display labels  ──────────────────────────────
label_map <- c(
  "hypertension"             = "Hypertension",
  "diabetes"                 = "Diabetes",
  "cmvigg"                   = "CMV IgG",
  "adulteduc"                = "Educational Attainment",
  "animal_naming46_z"        = "Age 46 - Verbal Fluency",
  "bmi_z"                    = "Body Mass Index",
  "bp_dia_z"                 = "Diastolic Blood Pressure",
  "bp_sys_z"                 = "Systolic Blood Pressure",
  "chol_std"                 = "Cholesterol",
  "hba1c_std"                = "HbA1c",
  "hdl_std"                  = "HDL",
  "igf1_std"                 = "IGF-1",
  "letter_correct46_z"       = "Age 46 - Processing (score)",
  "letter_speed46_z"         = "Age 46 - Processing (speed)",
  "log_crp_std"              = "log(CRP)",
  "log_frtin_std"            = "log(Ferritin)",
  "log_trig_std"             = "log(Triglycerides)",
  "maths_score10_z"          = "Age 10 - Maths",
  "maths_score16_z"          = "Age 16 - Maths",
  "matrices_score16_z"       = "Age 16 - Matrices",
  "rbc_std"                  = "Red Blood Cell Count",
  "reading_score_raw10_z"    = "Age 10 - Reading",
  "reading_score_total16_z"  = "Age 16 - Reading",
  "waisthip_z"               = "Waist/Hip Ratio",
  "wordlist_delayed46_z"     = "Age 46 - Verbal Memory (delayed)",
  "wordlist_immediate46_z"   = "Age 46 - Verbal Memory (immediate)",
  "vocab42_z"                = "Age 42 - Vocabulary",
  "literacy35_z"             = "Age 35 - Literacy",
  "numeracy35_z"             = "Age 35 - Numeracy"
)


c# ── Reshape to long - IPW only ────────────────────────────────────────────────
plot_df <- comparison_table %>%
  mutate(outcome_label = recode(outcome, !!!label_map)) %>%
  select(outcome, outcome_label, outcome_group, family,
         estimate  = ipw_est,
         conf.low  = ipw_low,
         conf.high = ipw_high,
         p.value   = ipw_p,
         p_fdr     = ipw_p_fdr)

# ── Helper ────────────────────────────────────────────────────────────────────
make_panel <- function(data, xlab, vline, tag,
                       x_limits = NULL, x_breaks = NULL,
                       manual_order = FALSE) {
  p <- ggplot(data,
              aes(x = estimate,
                  y = if (manual_order) outcome_label else fct_reorder(outcome_label, estimate))) +
    geom_vline(xintercept = vline, linetype = "dashed",
               colour = "grey40", linewidth = 0.5) +
    geom_errorbarh(aes(xmin = conf.low, xmax = conf.high),
                   height = 0, linewidth = 0.55,
                   colour = "black") +
    geom_point(size = 2.5, colour = "black", shape = 16) +
    labs(x = xlab, y = NULL, tag = tag) +
    my_custom_theme() +
    theme(
      legend.position   = "none",
      axis.text.y       = element_text(size = 11),
      plot.tag          = element_text(face = "bold", size = 11),
      plot.tag.position = "topleft"
    )

  if (!is.null(x_limits)) {
    p <- p +
      scale_x_continuous(breaks = x_breaks) +
      coord_cartesian(xlim = x_limits, clip = "on")
  }
  p
}

# ── Pull groups from CSV ──────────────────────────────────────────────────────
cardiometabolic <- comparison_table %>%
  filter(outcome_group == "cardiometabolic",
         outcome %in% c("hypertension", "diabetes")) %>%
  pull(outcome)

biomarkers_continuous <- comparison_table %>%
  filter(outcome_group == "biomarker",
         family == "continuous") %>%
  pull(outcome)

biomarkers_binary <- comparison_table %>%
  filter(outcome_group == "biomarker",
         family == "binomial") %>%
  pull(outcome)

cognitive <- comparison_table %>%
  filter(outcome_group == "cognitive",
         outcome != "adulteduc") %>%
  pull(outcome)



# ── Build panels ──────────────────────────────────────────────────────────────
p_A <- make_panel(
  plot_df %>%
    filter(outcome %in% cognitive) %>%
    mutate(outcome_label = factor(outcome_label, levels = cognitive_order)),
  xlab = "Beta (95% CI)", vline = 0, tag = "A",
  x_limits = c(-0.25, 0.1), x_breaks = seq(-0.2, 0.0, 0.1),
  manual_order = TRUE
)

p_B <- make_panel(
  plot_df %>% filter(outcome %in% cardiometabolic),
  xlab = "Odds Ratio (95% CI)", vline = 1, tag = "B",
  x_limits = c(0.9, 1.7), x_breaks = seq(0.9, 1.7, 0.2)
)

p_C <- make_panel(
  plot_df %>% filter(outcome %in% biomarkers_continuous),
  xlab = "Beta (95% CI)", vline = 0, tag = "C",
  x_limits = c(-0.2, 0.15), x_breaks = seq(-0.1, 0.1, 0.1)
)

p_D <- make_panel(
  plot_df %>% filter(outcome %in% biomarkers_binary),
  xlab = "Odds Ratio (95% CI)", vline = 1, tag = "D",
  x_limits = c(0.7, 1.1), x_breaks = seq(0.7, 1.1, 0.1)
)



# ── Assemble: A left, B/C/D right ────────────────────────────────────────────
left_col <- p_A  # single panel, no plot_layout needed

right_col <- (p_B / p_C / p_D) +
  plot_layout(heights = c(length(cardiometabolic),
                           length(biomarkers_continuous),
                           length(biomarkers_binary)))

combined <- (left_col | right_col) +
  plot_layout(widths = c(1.1, 1))

# ── Save ──────────────────────────────────────────────────────────────────────
ggsave("figure_ipw_results.pdf",
       plot = combined, width = 12, height = 11, device = cairo_pdf)

ggsave("figure_ipw_results.png",
       plot = combined, width = 12, height = 11, dpi = 300)

panels <- list(
  "A" = list(plot = p_A, width = 6, height = 4),
  "B" = list(plot = p_B, width = 6, height = 1.5),
  "C" = list(plot = p_C, width = 5, height = 4),
  "D" = list(plot = p_D, width = 5, height = 1.5)
)

for (nm in names(panels)) {
  ggsave(paste0("figure_panel_", nm, ".pdf"),
         plot   = panels[[nm]]$plot,
         width  = panels[[nm]]$width,
         height = panels[[nm]]$height,
         device = cairo_pdf)
  ggsave(paste0("figure_panel_", nm, ".png"),
         plot   = panels[[nm]]$plot,
         width  = panels[[nm]]$width,
         height = panels[[nm]]$height,
         dpi    = 300)
}